In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
def load_and_process_data(file_path) -> pd.DataFrame:
    """データの読み込みと前処理を行う関数"""

    # Excelファイルから売上データを読み込む
    df = pd.read_excel(file_path, sheet_name='売上データ')

    # 前処理
    df['売上'] = df['単価'] * df['数量']
    df['売上日'] = pd.to_datetime(df['売上日'])
    df['年月'] = df['売上日'].dt.strftime('%Y-%m')

    return df

In [3]:
def sum_sales_data(df: pd.DataFrame) -> pd.DataFrame:
    """集計を行う関数"""

    # 月ごとのカテゴリー別売上合計を集計
    pivot_table = pd.pivot_table(
        df,
        index='カテゴリー',
        columns='年月',
        values='売上',
        aggfunc='sum',
        fill_value=0
    )

    return pivot_table

In [4]:
def convert_to_prompt(df: pd.DataFrame, pivot_table: pd.DataFrame) -> str:
    """データをプロンプトに変換する関数"""

    sales_data_text = df.astype(str)
    pivot_table_text = pivot_table.astype(str)

    prompt_text = f"""
    売上データ：
    {sales_data_text}
    月ごとのカテゴリー別売上合計：
    {pivot_table_text}
    上記の「売上データ」と「月ごとのカテゴリー別売上合計」をもとに、カテゴリー毎の売上戦略を考案してください。
    """

    return prompt_text

In [5]:
def get_openai_response(client, prompt_text, model_name=MODEL_NAME) -> str:
    """OpenAI APIの呼び出しを行う関数"""

    role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt_text},
        ],
    )

    # APIの応答から出力テキストを抽出して返す
    return response.choices[0].message.content.strip()

In [6]:
def save_result_to_file(result, file_path="カテゴリー毎の売上戦略.md"):
    """結果をファイルに保存する関数"""
    with open(file_path, mode="w", encoding="utf-8") as file:
        file.write(result)

In [7]:
# ワークフロー（メイン処理）
def main():
    print("処理を開始します。")

    # 1.データの読み込みと前処理
    print("（1/5）データの読み込みと前処理")
    df = load_and_process_data('サンプルデータ.xlsx')

    # 2.集計
    print("（2/5）集計")
    pivot_table = sum_sales_data(df)

    # 3.プロンプト生成
    print("（3/5）プロンプト生成")
    prompt_text = convert_to_prompt(df, pivot_table)

    # 4.OpenAI APIからの応答を取得
    print("（4/5）OpenAI APIからの応答を取得")
    result = get_openai_response(client, prompt_text)

    # 5.結果をファイルに保存
    print("（5/5）結果をファイルに保存")
    save_result_to_file(result)
    print("分析結果を保存しました。")

In [8]:
main()

処理を開始します。
（1/5）データの読み込みと前処理
（2/5）集計
（3/5）プロンプト生成
（4/5）OpenAI APIからの応答を取得
（5/5）結果をファイルに保存
分析結果を保存しました。
